# [Cell 1] Cài đặt dependencies cho ETL cập nhật ảnh, Azure Public Storage & Hugging Face Hub

In [1]:
# [Cell 1] Cài đặt dependencies cho ETL cập nhật ảnh, Azure Public Storage & Hugging Face Hub
!pip install -q datasets huggingface_hub pandas pyarrow aiohttp nest_asyncio azure-storage-blob
print("✅ Hoàn tất cài đặt các thư viện cần thiết.")

✅ Hoàn tất cài đặt các thư viện cần thiết.


# [Cell 2] Nạp Secret Keys an toàn từ Google Colab Secrets

In [2]:
# [Cell 2] Nạp Secret Keys an toàn từ Google Colab Secrets
import os
from google.colab import userdata

HF_TOKEN = userdata.get("HF_WRITE_MODELS_TOKEN")
HF_USERNAME = userdata.get("HF_USERNAME")
HF_DATASET_REPO = f"{HF_USERNAME}/amazon-musical-instruments-clean"

try:
    AZURE_STORAGE_CONN_STR = userdata.get("AZURE_STORAGE_CONNECTION_STRING")
    print("🔑 Đã nạp AZURE_STORAGE_CONNECTION_STRING.")
except Exception:
    AZURE_STORAGE_CONN_STR = None
    print("⚠️ Không tìm thấy AZURE_STORAGE_CONNECTION_STRING -> Sẽ giữ URL Amazon đã validate.")

TARGET_CATEGORY = "Musical_Instruments"
PUBLIC_IMAGES_CONTAINER = "product-images"

os.makedirs("data/processed", exist_ok=True)
os.makedirs("data/images", exist_ok=True)

print(f"⚙️ Dataset Repo đích: {HF_DATASET_REPO}")
print(f"🪣 Public Storage Container: {PUBLIC_IMAGES_CONTAINER}")

🔑 Đã nạp AZURE_STORAGE_CONNECTION_STRING.
⚙️ Dataset Repo đích: cuongduongok/amazon-musical-instruments-clean
🪣 Public Storage Container: product-images


# [Cell 3] Tải web_products.parquet & training_products.parquet hiện có từ Hugging Face

In [3]:
# [Cell 3] Tải web_products.parquet & training_products.parquet hiện có từ Hugging Face
import pandas as pd
from huggingface_hub import hf_hub_download

print(f"⬇️ Đang tải dữ liệu sản phẩm hiện tại từ {HF_DATASET_REPO}...")

web_parquet_cached = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename="web_products.parquet",
    repo_type="dataset",
    token=HF_TOKEN,
)
train_parquet_cached = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename="training_products.parquet",
    repo_type="dataset",
    token=HF_TOKEN,
)

df_web_old = pd.read_parquet(web_parquet_cached)
df_train_old = pd.read_parquet(train_parquet_cached)

target_pids = set(df_train_old["product_id"].astype(str))
print(f"✅ Đã nạp {len(target_pids)} sản phẩm hiện có (giữ nguyên toàn bộ ID -> KHÔNG cần train lại SVD/Similarity).")
print("📋 Các cột hiện tại của training_products:", list(df_train_old.columns))

⬇️ Đang tải dữ liệu sản phẩm hiện tại từ cuongduongok/amazon-musical-instruments-clean...


web_products.parquet: reconstructing file:   0%|          |  0.00B /  268kB            

web_products.parquet: downloading bytes:           |  0.00B            

training_products.parquet: reconstructing file:   0%|          |  0.00B /  987kB            

training_products.parquet: downloading bytes:           |  0.00B            

✅ Đã nạp 1000 sản phẩm hiện có (giữ nguyên toàn bộ ID -> KHÔNG cần train lại SVD/Similarity).
📋 Các cột hiện tại của training_products: ['product_id', 'name', 'category', 'price', 'description', 'features', 'rating', 'rating_count', 'stock_quantity', 'content_raw', 'bought_together', 'thumbnail_url', 'large_url', 'hi_res_url']


# [Cell 4] Streaming Amazon-Reviews-2023 Metadata & Kiểm tra sống (1 hi_res, tối đa 3 large, 1 thumb)

In [5]:
# [Cell 4] Đọc trực tiếp Metadata Parquet từ Hugging Face & Kiểm tra sống (1 hi_res, tối đa 3 large, 1 thumb)
import asyncio
import aiohttp
import nest_asyncio
import pandas as pd
from huggingface_hub import HfApi, hf_hub_download

nest_asyncio.apply()

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    )
}

def clean_url_list(raw_list) -> list:
    if raw_list is None:
        return []
    if hasattr(raw_list, "tolist"):
        raw_list = raw_list.tolist()
    if not isinstance(raw_list, (list, tuple)):
        raw_list = [raw_list]
    seen = set()
    result = []
    for u in raw_list:
        if isinstance(u, str):
            u_clean = u.strip()
            if u_clean.startswith(("http://", "https://")) and u_clean not in seen:
                seen.add(u_clean)
                result.append(u_clean)
    return result

def extract_image_candidates(images_field) -> dict:
    hi_res, large, thumb = [], [], []
    if isinstance(images_field, dict):
        hi_res = clean_url_list(images_field.get("hi_res"))
        large = clean_url_list(images_field.get("large"))
        thumb = clean_url_list(images_field.get("thumb"))
    elif isinstance(images_field, (list, tuple)):
        for entry in images_field:
            if isinstance(entry, dict):
                hi_res.extend(clean_url_list(entry.get("hi_res")))
                large.extend(clean_url_list(entry.get("large")))
                thumb.extend(clean_url_list(entry.get("thumb")))
            elif isinstance(entry, str):
                large.extend(clean_url_list(entry))
    return {
        "hi_res": clean_url_list(hi_res),
        "large": clean_url_list(large),
        "thumb": clean_url_list(thumb),
    }

# 1. Tìm và tải trực tiếp các file Parquet của raw_meta_Musical_Instruments
print(f"⏳ [1/2] Đang tìm các file Metadata Parquet ({TARGET_CATEGORY}) trên Hugging Face...")
hf_api = HfApi(token=HF_TOKEN)
source_repo = "McAuley-Lab/Amazon-Reviews-2023"
meta_prefix = f"raw_meta_{TARGET_CATEGORY}/"

revision = "main"
repo_files = hf_api.list_repo_files(repo_id=source_repo, repo_type="dataset", revision=revision)
parquet_paths = sorted([f for f in repo_files if f.startswith(meta_prefix) and f.endswith(".parquet")])

# Fallback sang nhánh refs/convert/parquet nếu nhánh main không chứa file parquet trực tiếp
if not parquet_paths:
    revision = "refs/convert/parquet"
    repo_files = hf_api.list_repo_files(repo_id=source_repo, repo_type="dataset", revision=revision)
    parquet_paths = sorted([f for f in repo_files if f.startswith(meta_prefix) and f.endswith(".parquet")])

print(f"📂 Tìm thấy {len(parquet_paths)} file Parquet (revision={revision}): {parquet_paths}")

matched_images_meta = {}
for pq_file in parquet_paths:
    local_pq = hf_hub_download(
        repo_id=source_repo,
        filename=pq_file,
        repo_type="dataset",
        revision=revision,
        token=HF_TOKEN,
    )
    # Chỉ đọc đúng 2 cột parent_asin & images để tiết kiệm RAM và chạy trong vài giây
    df_meta_part = pd.read_parquet(local_pq, columns=["parent_asin", "images"])
    df_matched = df_meta_part[df_meta_part["parent_asin"].isin(target_pids)]

    for _, r in df_matched.iterrows():
        pid = str(r["parent_asin"])
        if pid not in matched_images_meta:
            matched_images_meta[pid] = extract_image_candidates(r["images"])

    print(f"   ✓ Đã khớp {len(matched_images_meta)}/{len(target_pids)} sản phẩm sau khi đọc {pq_file}")
    if len(matched_images_meta) >= len(target_pids):
        break

# 2. Ping bất đồng bộ kiểm tra link ảnh sống
async def check_url_alive(session: aiohttp.ClientSession, url: str, sem: asyncio.Semaphore) -> bool:
    if not url or not url.startswith(("http://", "https://")):
        return False
    async with sem:
        try:
            async with session.head(url, headers=HEADERS, timeout=aiohttp.ClientTimeout(total=5), allow_redirects=True) as resp:
                if resp.status == 200 and "image" in resp.headers.get("Content-Type", "").lower():
                    return True
                if resp.status in (403, 405):
                    async with session.get(url, headers=HEADERS, timeout=aiohttp.ClientTimeout(total=5), allow_redirects=True) as get_resp:
                        return get_resp.status == 200 and "image" in get_resp.headers.get("Content-Type", "").lower()
            return False
        except Exception:
            return False

async def validate_all_candidates():
    urls_to_check = set()
    candidates_by_pid = {}

    for _, row in df_train_old.iterrows():
        pid = str(row["product_id"])
        old_url = str(row.get("thumbnail_url") or row.get("image_url") or "")
        imgs = matched_images_meta.get(pid, {"hi_res": [], "large": [], "thumb": []})

        hi_cands = imgs["hi_res"][:3]
        lg_cands = imgs["large"][:5]
        th_cands = imgs["thumb"][:3]

        if old_url.startswith("http"):
            if not lg_cands:
                lg_cands.append(old_url)
            if not th_cands:
                th_cands.append(old_url)

        candidates_by_pid[pid] = {"hi_res": hi_cands, "large": lg_cands, "thumb": th_cands}
        urls_to_check.update(hi_cands + lg_cands + th_cands)

    print(f"🌐 [2/2] Đang ping kiểm tra sống {len(urls_to_check)} URL ảnh ứng viên...")
    url_list = list(urls_to_check)
    sem = asyncio.Semaphore(80)
    connector = aiohttp.TCPConnector(limit=80, ssl=False)
    async with aiohttp.ClientSession(connector=connector) as session:
        flags = await asyncio.gather(*(check_url_alive(session, u, sem) for u in url_list))
    alive_map = dict(zip(url_list, flags))

    resolved = {}
    for pid, cands in candidates_by_pid.items():
        live_large = [u for u in cands["large"] if alive_map.get(u)][:3]
        live_hi = next((u for u in cands["hi_res"] if alive_map.get(u)), None)
        live_thumb = next((u for u in cands["thumb"] if alive_map.get(u)), None)

        fallback = live_large[0] if live_large else (live_hi or live_thumb or "")
        if not live_hi:
            live_hi = fallback
        if not live_large and fallback:
            live_large = [fallback]
        if not live_thumb:
            live_thumb = fallback

        resolved[pid] = {
            "hi_res_url": live_hi,
            "large_urls": live_large,
            "thumb_url": live_thumb,
        }
    return resolved

loop = asyncio.get_event_loop()
source_images_map = loop.run_until_complete(validate_all_candidates())

count_3_large = sum(1 for v in source_images_map.values() if len(v["large_urls"]) == 3)
print(f"🖼️ Hoàn tất validate: {len(source_images_map)}/{len(df_train_old)} SP đủ ảnh (trong đó {count_3_large} SP có đủ 3 ảnh large).")

⏳ [1/2] Đang tìm các file Metadata Parquet (Musical_Instruments) trên Hugging Face...
📂 Tìm thấy 2 file Parquet (revision=main): ['raw_meta_Musical_Instruments/full-00000-of-00002.parquet', 'raw_meta_Musical_Instruments/full-00001-of-00002.parquet']


raw_meta_Musical_Instruments/full-00000-(…): reconstructing file:   0%|          |  0.00B /  124MB            

raw_meta_Musical_Instruments/full-00000-(…): downloading bytes:           |  0.00B            

   ✓ Đã khớp 1000/1000 sản phẩm sau khi đọc raw_meta_Musical_Instruments/full-00000-of-00002.parquet
🌐 [2/2] Đang ping kiểm tra sống 8621 URL ảnh ứng viên...
🖼️ Hoàn tất validate: 1000/1000 SP đủ ảnh (trong đó 722 SP có đủ 3 ảnh large).


# [Cell 5] Tải song song toàn bộ ảnh & Upload lên Public Container `product-images` trên Azure Storage

In [6]:
# [Cell 5] Tải song song toàn bộ ảnh & Upload lên Public Container `product-images` trên Azure Storage
import asyncio
import aiohttp
from azure.core.exceptions import ResourceExistsError
from azure.storage.blob import BlobServiceClient, ContentSettings, PublicAccess

# 1. Tải toàn bộ ảnh về data/images/{product_id}/{hi_res.jpg | large_0..2.jpg | thumb.jpg}
download_tasks = []
for pid, imgs in source_images_map.items():
    if imgs["hi_res_url"]:
        rel = f"{pid}/hi_res.jpg"
        download_tasks.append((pid, rel, imgs["hi_res_url"], os.path.join("data/images", rel)))
    for i, lg_url in enumerate(imgs["large_urls"][:3]):
        rel = f"{pid}/large_{i}.jpg"
        download_tasks.append((pid, rel, lg_url, os.path.join("data/images", rel)))
    if imgs["thumb_url"]:
        rel = f"{pid}/thumb.jpg"
        download_tasks.append((pid, rel, imgs["thumb_url"], os.path.join("data/images", rel)))

async def download_img(session: aiohttp.ClientSession, url: str, save_path: str, sem: asyncio.Semaphore) -> bool:
    if os.path.exists(save_path) and os.path.getsize(save_path) > 0:
        return True
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    async with sem:
        try:
            async with session.get(url, headers=HEADERS, timeout=aiohttp.ClientTimeout(total=10)) as resp:
                if resp.status == 200:
                    content = await resp.read()
                    if content:
                        with open(save_path, "wb") as f:
                            f.write(content)
                        return True
            return False
        except Exception:
            return False

async def run_downloads():
    sem = asyncio.Semaphore(80)
    connector = aiohttp.TCPConnector(limit=80, ssl=False)
    async with aiohttp.ClientSession(connector=connector) as session:
        flags = await asyncio.gather(
            *(download_img(session, url, path, sem) for _, _, url, path in download_tasks)
        )
    return [(pid, rel, path) for (pid, rel, _, path), ok in zip(download_tasks, flags) if ok]

print(f"⬇️ Đang tải song song {len(download_tasks)} file ảnh về Colab...")
downloaded_files = asyncio.get_event_loop().run_until_complete(run_downloads())
print(f"✅ Đã tải thành công {len(downloaded_files)}/{len(download_tasks)} file ảnh.")

# 2. Đẩy lên Azure Blob Storage với PublicAccess.Blob (nếu có AZURE_STORAGE_CONNECTION_STRING)
if AZURE_STORAGE_CONN_STR:
    blob_service = BlobServiceClient.from_connection_string(AZURE_STORAGE_CONN_STR)
    container_client = blob_service.get_container_client(PUBLIC_IMAGES_CONTAINER)
    try:
        container_client.create_container(public_access=PublicAccess.Blob)
        print(f"🪣 Đã tạo public container mới: '{PUBLIC_IMAGES_CONTAINER}' (PublicAccess.Blob)")
    except ResourceExistsError:
        container_client.set_container_access_policy(
            signed_identifiers={}, public_access=PublicAccess.Blob
        )
        print(f"🪣 Container '{PUBLIC_IMAGES_CONTAINER}' đã tồn tại -> Đã bật PublicAccess.Blob.")

    print(f"☁️ Đang upload {len(downloaded_files)} ảnh lên Azure Public Storage...")
    uploaded_url_map = {}
    content_settings = ContentSettings(
        content_type="image/jpeg",
        cache_control="public, max-age=31536000",
    )

    for idx, (pid, rel_path, local_path) in enumerate(downloaded_files, start=1):
        blob_client = container_client.get_blob_client(rel_path)
        with open(local_path, "rb") as f:
            blob_client.upload_blob(f, overwrite=True, content_settings=content_settings)
        uploaded_url_map[rel_path] = blob_client.url
        if idx % 500 == 0 or idx == len(downloaded_files):
            print(f"   ☁️ Uploaded {idx}/{len(downloaded_files)} blobs...")

    final_images_map = {}
    for pid, orig in source_images_map.items():
        hi_url = uploaded_url_map.get(f"{pid}/hi_res.jpg") or orig["hi_res_url"]
        lg_urls = [
            uploaded_url_map.get(f"{pid}/large_{i}.jpg") or orig_lg
            for i, orig_lg in enumerate(orig["large_urls"][:3])
        ]
        th_url = uploaded_url_map.get(f"{pid}/thumb.jpg") or orig["thumb_url"]
        final_images_map[pid] = {
            "hi_res_url": hi_url,
            "large_urls": lg_urls,
            "thumb_url": th_url,
        }
    print("🎉 Hoàn tất cập nhật URL sang Azure Public Blob Storage!")
else:
    final_images_map = source_images_map
    print("ℹ️ Dùng URL ảnh gốc Amazon đã validate.")

⬇️ Đang tải song song 4522 file ảnh về Colab...
✅ Đã tải thành công 4522/4522 file ảnh.
🪣 Đã tạo public container mới: 'product-images' (PublicAccess.Blob)
☁️ Đang upload 4522 ảnh lên Azure Public Storage...
   ☁️ Uploaded 500/4522 blobs...
   ☁️ Uploaded 1000/4522 blobs...
   ☁️ Uploaded 1500/4522 blobs...
   ☁️ Uploaded 2000/4522 blobs...
   ☁️ Uploaded 2500/4522 blobs...
   ☁️ Uploaded 3000/4522 blobs...
   ☁️ Uploaded 3500/4522 blobs...
   ☁️ Uploaded 4000/4522 blobs...
   ☁️ Uploaded 4500/4522 blobs...
   ☁️ Uploaded 4522/4522 blobs...
🎉 Hoàn tất cập nhật URL sang Azure Public Blob Storage!


# [Cell 6] Ghi đè Parquet, CSV, JSON, SQL Seed & Overwrite sạch lên Hugging Face Hub

In [7]:
# [Cell 6] Ghi đè Parquet, CSV, JSON, SQL Seed & Overwrite sạch lên Hugging Face Hub
import json
from huggingface_hub import HfApi, create_repo

df_train_new = df_train_old.copy()

# Lưu URL gốc + URL serving chính thức
df_train_new["source_hi_res_url"] = df_train_new["product_id"].map(lambda pid: source_images_map[str(pid)]["hi_res_url"])
df_train_new["source_large_urls"] = df_train_new["product_id"].map(lambda pid: source_images_map[str(pid)]["large_urls"])
df_train_new["source_thumb_url"] = df_train_new["product_id"].map(lambda pid: source_images_map[str(pid)]["thumb_url"])

df_train_new["hi_res_url"] = df_train_new["product_id"].map(lambda pid: final_images_map[str(pid)]["hi_res_url"])
df_train_new["large_urls"] = df_train_new["product_id"].map(lambda pid: final_images_map[str(pid)]["large_urls"])
df_train_new["thumb_url"] = df_train_new["product_id"].map(lambda pid: final_images_map[str(pid)]["thumb_url"])

# Giữ tương thích ngược cho thumbnail_url & image_url
df_train_new["thumbnail_url"] = df_train_new["thumb_url"]
df_train_new["image_url"] = df_train_new.apply(
    lambda r: r["hi_res_url"] or (r["large_urls"][0] if r["large_urls"] else r["thumb_url"]),
    axis=1,
)

web_cols = [
    c for c in [
        "product_id", "name", "category", "price",
        "image_url", "hi_res_url", "large_urls", "thumb_url", "thumbnail_url",
        "description", "rating", "rating_count", "stock_quantity"
    ] if c in df_train_new.columns
]
df_web_new = df_train_new[web_cols].copy()

# 1. Xuất các file dữ liệu mới
df_web_new.to_parquet("data/processed/web_products.parquet", index=False)
df_train_new.to_parquet("data/processed/training_products.parquet", index=False)
df_web_new.to_csv("data/processed/web_products.csv", index=False, encoding="utf-8-sig")
df_web_new.head(50).to_json("data/processed/web_products_sample_50.json", orient="records", indent=2, force_ascii=False)

# 2. Sinh seed_products.sql tự nâng cấp schema + UPSERT ghi đè ảnh mới
sql_lines = [
    "CREATE TABLE IF NOT EXISTS products (",
    "    id VARCHAR(50) PRIMARY KEY,",
    "    name VARCHAR(255) NOT NULL,",
    "    category VARCHAR(100),",
    "    price NUMERIC(10, 2) NOT NULL DEFAULT 0.00,",
    "    image_url TEXT,",
    "    hi_res_url TEXT,",
    "    large_urls JSONB NOT NULL DEFAULT '[]'::jsonb,",
    "    thumb_url TEXT,",
    "    stock_quantity INT NOT NULL DEFAULT 100 CHECK (stock_quantity >= 0),",
    "    description TEXT,",
    "    is_active BOOLEAN NOT NULL DEFAULT TRUE,",
    "    created_at TIMESTAMP WITH TIME ZONE DEFAULT CURRENT_TIMESTAMP,",
    "    updated_at TIMESTAMP WITH TIME ZONE DEFAULT CURRENT_TIMESTAMP",
    ");",
    "",
    "ALTER TABLE products",
    "    ADD COLUMN IF NOT EXISTS hi_res_url TEXT,",
    "    ADD COLUMN IF NOT EXISTS large_urls JSONB NOT NULL DEFAULT '[]'::jsonb,",
    "    ADD COLUMN IF NOT EXISTS thumb_url TEXT,",
    "    ADD COLUMN IF NOT EXISTS is_active BOOLEAN NOT NULL DEFAULT TRUE,",
    "    ADD COLUMN IF NOT EXISTS updated_at TIMESTAMP WITH TIME ZONE DEFAULT CURRENT_TIMESTAMP;",
    "",
    "\\set QUIET on",
    "\\echo '>>> [INIT DB] Dang chay file seed_products.sql (Multi-Resolution Images)...'",
    "",
]

for _, r in df_web_new.iterrows():
    pid = str(r["product_id"]).replace("'", "''")
    name = str(r["name"]).replace("'", "''")[:255]
    cat = str(r.get("category") or "").replace("'", "''")[:100]
    price = float(r.get("price") or 0.0)
    img_url = str(r.get("image_url") or "").replace("'", "''")
    hi_res = str(r.get("hi_res_url") or "").replace("'", "''")
    large_json = json.dumps(list(r.get("large_urls") or []), ensure_ascii=False).replace("'", "''")
    thumb = str(r.get("thumb_url") or "").replace("'", "''")
    stock = int(r.get("stock_quantity") or 100)
    desc = str(r.get("description") or "").replace("'", "''")

    sql_lines.append(
        "INSERT INTO products "
        "(id, name, category, price, image_url, hi_res_url, large_urls, thumb_url, stock_quantity, description) "
        f"VALUES ('{pid}', '{name}', '{cat}', {price}, '{img_url}', '{hi_res}', "
        f"'{large_json}'::jsonb, '{thumb}', {stock}, '{desc}') "
        "ON CONFLICT (id) DO UPDATE SET "
        "name = EXCLUDED.name, "
        "category = EXCLUDED.category, "
        "price = EXCLUDED.price, "
        "image_url = EXCLUDED.image_url, "
        "hi_res_url = EXCLUDED.hi_res_url, "
        "large_urls = EXCLUDED.large_urls, "
        "thumb_url = EXCLUDED.thumb_url, "
        "description = EXCLUDED.description, "
        "updated_at = CURRENT_TIMESTAMP;"
    )

with open("data/processed/seed_products.sql", "w", encoding="utf-8") as f:
    f.write("\n".join(sql_lines) + "\n")

print("💾 Đã ghi đè dữ liệu mới tại data/processed/.")

# 3. Đẩy ghi đè lên Hugging Face Dataset (xóa cấu trúc images cũ trước khi upload)
api = HfApi(token=HF_TOKEN)
create_repo(repo_id=HF_DATASET_REPO, repo_type="dataset", token=HF_TOKEN, exist_ok=True)

print(f"🚀 Đang ghi đè data/processed/ lên Hugging Face: {HF_DATASET_REPO}...")
api.upload_folder(
    folder_path="data/processed",
    repo_id=HF_DATASET_REPO,
    repo_type="dataset",
    commit_message="Overwrite products dataset with multi-resolution images (1 hi_res, 3 large, 1 thumb)",
)

print(f"🚀 Đang ghi đè thư mục ảnh mới lên {HF_DATASET_REPO}/images...")
api.upload_folder(
    folder_path="data/images",
    path_in_repo="images",
    repo_id=HF_DATASET_REPO,
    repo_type="dataset",
    delete_patterns=["images/**"],
    commit_message="Replace backup images with multi-resolution structure (hi_res, large_0..2, thumb)",
)

print(f"🎉 Hoàn tất ghi đè lên Hugging Face: https://huggingface.co/datasets/{HF_DATASET_REPO}")

💾 Đã ghi đè dữ liệu mới tại data/processed/.
🚀 Đang ghi đè data/processed/ lên Hugging Face: cuongduongok/amazon-musical-instruments-clean...
🚀 Đang ghi đè thư mục ảnh mới lên cuongduongok/amazon-musical-instruments-clean/images...
🎉 Hoàn tất ghi đè lên Hugging Face: https://huggingface.co/datasets/cuongduongok/amazon-musical-instruments-clean


# [Cell 7] Đóng gói toàn bộ dữ liệu mới (Parquet, CSV, JSON mẫu, seed_products.sql) tải về máy

In [8]:
# [Cell 7] Đóng gói toàn bộ dữ liệu mới (Parquet, CSV, JSON mẫu, seed_products.sql) tải về máy
import zipfile
from google.colab import files

zip_filename = "web_data_multi_images_updated.zip"
files_to_pack = [
    "data/processed/web_products.parquet",
    "data/processed/training_products.parquet",
    "data/processed/web_products.csv",
    "data/processed/web_products_sample_50.json",
    "data/processed/seed_products.sql",
]

with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zipf:
    for fp in files_to_pack:
        if os.path.exists(fp):
            zipf.write(fp, arcname=os.path.basename(fp))
            print(f"📦 Đã đóng gói: {os.path.basename(fp)}")

print(f"\n⬇️ Đang tải {zip_filename} về máy...")
files.download(zip_filename)

📦 Đã đóng gói: web_products.parquet
📦 Đã đóng gói: training_products.parquet
📦 Đã đóng gói: web_products.csv
📦 Đã đóng gói: web_products_sample_50.json
📦 Đã đóng gói: seed_products.sql

⬇️ Đang tải web_data_multi_images_updated.zip về máy...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>